# Exploração · `bronze.medidores`

Notebook de **investigação**, fora do pipeline. Descobre os problemas de qualidade de `medidores` e a regra de limpeza de cada um, antes de escrever a Silver.


| # | Seção | Pergunta |
|---|---|---|
| 1 | Visão geral | Colunas, tipos e volume |
| 2 | Nulos | Quais colunas têm vazios? |
| 3 | Duplicatas | As 513 linhas repetidas são cópias exatas? |
| 4 | Formato dos ids | `id_medidor` e `id_uc` seguem o padrão? Há UCs órfãs? |
| 5 | Categorias | `tipo`, `modelo` e `status` têm variações de grafia? |
| 6 | Formatos de data | Quantos formatos e sentinelas há em cada coluna de data? |
| 7 | Ordem dia/mês de `data_instalacao` | Qual formato é dia primeiro e qual é mês primeiro? |
| 8 | Status × datas | `status` é coerente com `data_retirada`? Datas impossíveis? |
| 9 | Medidores por UC | UCs sem medidor ativo ou com mais de um |
| 10 | Cruzamento com leituras | A Silver de medidores preserva os medidores usados em leituras? |
| 11 | Resumo | Regras propostas para a Silver |


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

medidores = spark.table(f"{bronze_schema}.medidores")
ucs_bronze = spark.table(f"{bronze_schema}.unidades_consumidoras")

def n_br(n):
    return f"{n:,}".replace(",", ".")

## 1. Visão geral

**O que explora:** colunas, tipos e uma amostra. Na Bronze todas as colunas são `string`, mais as colunas de auditoria (`source_file`, `ingestion_timestamp`).

**Achado:** 8 colunas de origem, todas `string`, mais as 2 de auditoria. 51.812 linhas.

In [0]:
medidores.printSchema()
display(medidores.limit(5))
print(f"{n_br(medidores.count())} linhas")

## 2. Nulos

**O que explora:** nulos reais e textos vazios (só espaços) por coluna.

**Achado (tabela bruta, com cópias):**
- `data_retirada`: 31.796 nulos reais (medidores ativos sem data; os ativos com a sentinela `9999-12-31` não contam aqui).
- `data_ultima_manutencao`: 51.812 nulos, ou seja, **100% vazia** (log M1).
- Nenhuma outra coluna tem nulo, e nenhuma tem texto vazio ou só com espaços.

In [0]:
cols_origem = [c for c in medidores.columns if c not in ("source_file", "ingestion_timestamp")]

display(medidores.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols_origem]
))
# texto vazio ou só com espaços (não é nulo real)
display(medidores.select(
    [F.sum((F.trim(F.col(c)) == "").cast("int")).alias(c) for c in cols_origem]
))

## 3. Duplicatas

**O que explora:** se `id_medidor` se repete e se as repetições são idênticas (sem as colunas de auditoria). Se *linhas distintas = ids distintos*, as repetições são cópias exatas e `dropDuplicates(["id_medidor"])` é seguro.

**Achado:**
- 51.812 linhas para 51.299 ids distintos. As linhas distintas (sem auditoria) também são 51.299, e nenhum id tem conteúdo diferente entre as repetições: são **cópias exatas**.
- São **513 ids que aparecem exatamente 2 vezes** (nenhum aparece 3 ou mais).
- **Decisão:** `dropDuplicates(["id_medidor"])`, antes de qualquer outro tratamento (log M3). A partir daqui, as contagens usam `med_d` (51.299 linhas).

In [0]:
print("linhas:                         ", n_br(medidores.count()))
print("id_medidor distintos:           ", n_br(medidores.select("id_medidor").distinct().count()))
print("linhas distintas (sem auditoria):", n_br(medidores.select(cols_origem).distinct().count()))

# Se não forem cópias exatas: mostra ids repetidos com conteúdo diferente
repetidos = medidores.select(cols_origem).distinct().groupBy("id_medidor").count().filter("count > 1")
print("ids com conteúdo diferente entre as repetições:", n_br(repetidos.count()))
display(medidores.join(repetidos.select("id_medidor"), "id_medidor").orderBy("id_medidor").limit(10))

In [0]:
# As cópias vêm em bloco (parte do arquivo duplicada) ou espalhadas? Mostra alguns ids repetidos
display(medidores.groupBy("id_medidor").count().filter("count > 1")
        .groupBy("count").agg(F.count("*").alias("qtd_ids")))

## 4. Formato dos ids e UCs órfãs

**O que explora:** `id_medidor` deve ser `MD` + 6 dígitos e `id_uc` deve ser `UC` + 6 dígitos (padrão usado em leituras e faturas). Verifica também se todo `id_uc` existe em `unidades_consumidoras` (que tem 1 linha de cabeçalho repetido, removida aqui, log U1).

**Achado:** todos os 51.299 `id_medidor` estão no padrão `MD` + 6 dígitos e todos os `id_uc` no padrão `UC` + 6 dígitos. **Todos os `id_uc` existem** em `unidades_consumidoras`: não há medidores órfãos. (Diferente de leituras e faturas, que tinham órfãs com ids de 7 dígitos.)

In [0]:
med_d = medidores.dropDuplicates(["id_medidor"])

display(med_d.groupBy(
    F.col("id_medidor").rlike("^MD[0-9]{6}$").alias("id_medidor_no_padrao"),
    F.col("id_uc").rlike("^UC[0-9]{6}$").alias("id_uc_no_padrao"),
).count())

ids_uc = (ucs_bronze.filter(F.col("id_uc") != "id_uc")
          .select("id_uc").distinct().withColumn("uc_existe", F.lit(True)))

display(med_d.join(ids_uc, "id_uc", "left")
        .groupBy(F.coalesce("uc_existe", F.lit(False)).alias("uc_existe_no_cadastro")).count())

## 5. Categorias

**O que explora:** valores distintos de `tipo`, `modelo` e `status` como vieram, e depois de `upper(trim(...))`. Se o número de valores cai depois da padronização, havia variações de grafia ou espaços.

**Achado:**
- **`tipo`:** 10 grafias para 3 tipos reais. 5.297 linhas (10,3%) fora do padrão. Só `upper(trim)` não resolve: também é preciso tirar acento, tirar hífen e trocar `SMART METER` por `SMART` (log M5).
  - `ELETRONICO`: `ELETRONICO` (23.685), `Eletronico` (1.350), `eletrônico` (1.304)
  - `ELETROMECANICO`: `ELETROMECANICO` (14.703), `ELETRO-MECANICO` (601), `Eletromecanico` (598), `eletromecânico` (591)
  - `SMART`: `SMART` (7.614), `smart meter` (439), `Smart` (414)
- **`modelo`:** 9 modelos reais (`EL-2200`, `EL-3000`, `NX-700`, `EM-1200`, `EM-1400`, `CG-500`, `SM-4100`, `IQ-900`, `SM-4500`), com espaços no início e/ou no fim em 3.078 linhas (6,0%). `trim` resolve (log M6).
- **`status`:** limpo. Só `ATIVO` (45.000) e `RETIRADO` (6.299).
- **Modelo × tipo:** depois de padronizar, cada modelo tem um único tipo (CG/EM = eletromecânico, EL/NX = eletrônico, IQ/SM = smart). As "3 grafias por modelo" da tabela abaixo são só variações de escrita. Vira uma checagem na Silver.

In [0]:
for c in ["tipo", "modelo", "status"]:
    brutos = med_d.select(c).distinct().count()
    padronizados = med_d.select(F.upper(F.trim(F.col(c)))).distinct().count()
    print(f"{c}: {brutos} valores brutos -> {padronizados} depois de upper(trim)")
    display(med_d.groupBy(c).count().orderBy(F.desc("count")))

In [0]:
# Modelo x tipo: cada modelo deveria ter um único tipo
display(med_d.groupBy(F.upper(F.trim("modelo")).alias("modelo"))
        .agg(F.countDistinct(F.upper(F.trim("tipo"))).alias("tipos_distintos"),
             F.collect_set(F.upper(F.trim("tipo"))).alias("tipos"))
        .orderBy("modelo"))

## 6. Formatos de data

**O que explora:** troca cada dígito por `9` para mostrar os formatos de cada coluna de data, incluindo a sentinela `9999-12-31` (que aparece como `9999-99-99`, então é contada à parte).

**Achado:**
- **`data_instalacao`:** ISO (47.605) e três formatos não ISO: `99-99-9999` (1.250), `99/99/9999` (1.235) e `99.99.9999` (1.209). São 3.694 linhas (7,2%) fora do ISO (log M4). A ordem dia/mês é conferida na seção 7.
- **`data_retirada`:** nulo (31.500), sentinela `9999-12-31` (13.500) e ISO (6.299). Não há formatos mistos nas datas reais (log M2).
- **`data_ultima_manutencao`:** 100% nula (log M1).

In [0]:
for c in ["data_instalacao", "data_retirada", "data_ultima_manutencao"]:
    print(c)
    display(med_d
        .withColumn("padrao", F.when(F.col(c) == "9999-12-31", "SENTINELA 9999-12-31")
                               .otherwise(F.regexp_replace(F.coalesce(F.col(c), F.lit("<nulo>")), "[0-9]", "9")))
        .groupBy("padrao").count().orderBy(F.desc("count")))

## 7. Ordem dia/mês de `data_instalacao`


**7.1 Partes maiores que 12.** Se a primeira parte passa de 12, ela só pode ser dia, e se a segunda passa de 12, ela só pode ser dia. Um único caso já decide a ordem do padrão.

**7.2 Gabarito pela data de ligação da UC.** O primeiro medidor de uma UC tende a ser instalado na ligação (ex.: `MD500000` e `UC100000`, ambos em 2011-12-20). Primeiro mede-se quanto isso vale nos medidores em ISO, depois testa-se cada formato candidato nos não ISO.

**7.3 Gabarito pela troca de medidor.** Nas trocas de `silver.leituras`, o medidor novo é instalado no mês da troca (a exploração de leituras confirmou isso nos medidores em ISO).

**Achado:** hífen = **mês primeiro** (`MM-dd-yyyy`); ponto e barra = **dia primeiro** (`dd.MM.yyyy`, `dd/MM/yyyy`). É o mesmo mapeamento de `leituras` e `faturas`. Os três testes concordam:
- **7.1:** no hífen, 755 linhas têm a 2ª parte > 12 (só pode ser dia); na barra e no ponto, 723 e 726 têm a 1ª parte > 12. Nenhuma contradição.
- **7.2:** nos medidores em ISO, a instalação é igual à ligação da UC em 87,6% (41.678 de 47.605). Nos não ISO, o formato proposto bate com a ligação em 1.106 de 1.250, 1.066 de 1.209 e 1.060 de 1.235 (~87%, a mesma taxa do ISO), e converte 100%. A ordem trocada bate em no máximo 45 e deixa de converter 723 a 755 linhas.
- **7.3:** nas trocas de medidor, o mês de instalação é o mês da troca em 142 de 142, 142 de 142 e 173 de 173. A ordem trocada bate em no máximo 3.

In [0]:
nao_iso = (med_d
    .withColumn("padrao", F.regexp_replace("data_instalacao", "[0-9]", "9"))
    .filter(F.col("padrao").isin("99-99-9999", "99.99.9999", "99/99/9999"))
    .withColumn("p1", F.substring("data_instalacao", 1, 2).cast("int"))
    .withColumn("p2", F.substring("data_instalacao", 4, 2).cast("int")))

# 7.1 Se p1 > 12, a 1ª parte é dia; se p2 > 12, a 2ª parte é dia
display(nao_iso.groupBy("padrao").agg(
    F.count("*").alias("linhas"),
    F.sum((F.col("p1") > 12).cast("int")).alias("primeira_parte_maior_12"),
    F.sum((F.col("p2") > 12).cast("int")).alias("segunda_parte_maior_12"),
    F.sum(((F.col("p1") <= 12) & (F.col("p2") <= 12)).cast("int")).alias("ambiguas"),
))

In [0]:
# Funções de conversão: cada padrão com cada formato candidato
candidatos = {
    "99-99-9999": ["MM-dd-yyyy", "dd-MM-yyyy"],
    "99.99.9999": ["dd.MM.yyyy", "MM.dd.yyyy"],
    "99/99/9999": ["dd/MM/yyyy", "MM/dd/yyyy"],
}

def converte(col, fmt):
    return F.expr(f"to_date(try_to_timestamp({col}, '{fmt}'))")

# 7.2 Gabarito: data de ligação da UC (só UCs com ligação em ISO)
uc_lig = (ucs_bronze.filter(F.col("id_uc") != "id_uc")
          .dropDuplicates(["id_uc"])
          .select("id_uc", converte("data_ligacao", "yyyy-MM-dd").alias("ligacao"))
          .filter("ligacao IS NOT NULL"))

# Quanto o gabarito vale nos medidores em ISO?
iso = (med_d.filter(F.col("data_instalacao").rlike("^[0-9]{4}-[0-9]{2}-[0-9]{2}$"))
       .withColumn("instalacao", converte("data_instalacao", "yyyy-MM-dd"))
       .join(uc_lig, "id_uc"))
print("medidores em ISO com UC de ligação conhecida:", n_br(iso.count()))
print("  instalação = ligação (mesmo dia): ", n_br(iso.filter("instalacao = ligacao").count()))

# Nos não ISO: qual formato candidato bate com a ligação?
nao_iso_lig = nao_iso.join(uc_lig, "id_uc")
for padrao, fmts in candidatos.items():
    sub = nao_iso_lig.filter(F.col("padrao") == padrao)
    for fmt in fmts:
        conv = sub.withColumn("instalacao", converte("data_instalacao", fmt))
        print(f"{padrao} como {fmt}: linhas {n_br(sub.count())} | "
              f"não converte {n_br(conv.filter('instalacao IS NULL').count())} | "
              f"bate com ligação {n_br(conv.filter('instalacao = ligacao').count())}")

In [0]:
# 7.3 Gabarito: mês da troca de medidor em silver.leituras (precisa ter rodado 02_silver_leituras)
trocas = (spark.table(f"{silver_schema}.leituras").filter("troca_medidor")
          .select("id_medidor", F.col("mes_referencia").alias("mes_troca")))

nao_iso_troca = nao_iso.join(trocas, "id_medidor")
print("medidores não ISO que aparecem como medidor novo numa troca:", n_br(nao_iso_troca.count()))
for padrao, fmts in candidatos.items():
    sub = nao_iso_troca.filter(F.col("padrao") == padrao)
    for fmt in fmts:
        conv = sub.withColumn("mes_inst", F.date_format(converte("data_instalacao", fmt), "yyyy-MM"))
        print(f"{padrao} como {fmt}: linhas {n_br(sub.count())} | "
              f"mês de instalação = mês da troca {n_br(conv.filter('mes_inst = mes_troca').count())}")

## 8. Status × datas

**O que explora:** coerência entre `status` e `data_retirada` (já visto: ATIVO sem data ou com sentinela; RETIRADO com data real), e datas impossíveis: retirada antes da instalação, instalação no futuro (depois de 2026-06, fim da extração) ou muito antiga.

**Já registrado:** M2 (sentinela `9999-12-31` em 13.649 medidores ATIVOS).

**Achado:**
- `status` coerente com `data_retirada`: ATIVO vazia (31.500), ATIVO com sentinela (13.500), RETIRADO com data real (6.299). Nenhum ATIVO com data real e nenhum RETIRADO sem data.
- Nenhuma retirada antes da instalação e nenhuma data depois de 2026-06-30 (nas datas ISO; as 3.694 não ISO aparecem como `null` no histograma).
- **Anos de instalação:** cerca de 1.700 por ano de 1998 a 2021, quase nada em 2022–2024 (44, 100 e 93) e **3.416 em 2025 e 2.519 em 2026**. São os medidores novos da **concentração de trocas de medidor entre set/2025 e mar/2026** (6.299 trocas em `silver.leituras`, o mesmo número de RETIRADOS). Ponto para o diagnóstico da receita: a campanha cai justamente nos últimos 12 meses que o diretor compara.

In [0]:
display(med_d.groupBy(
    F.upper(F.trim("status")).alias("status"),
    F.when(F.col("data_retirada").isNull(), "vazia")
     .when(F.col("data_retirada") == "9999-12-31", "sentinela 9999-12-31")
     .otherwise("data real").alias("situacao_retirada"),
).count().orderBy("status", "situacao_retirada"))

In [0]:
# Datas convertidas só nos formatos já confirmados (ISO); os demais ficam para depois da seção 7
datas = (med_d
    .withColumn("instalacao", converte("data_instalacao", "yyyy-MM-dd"))
    .withColumn("retirada", F.when(F.col("data_retirada") != "9999-12-31", converte("data_retirada", "yyyy-MM-dd"))))

print("retirada antes da instalação:", n_br(datas.filter("retirada < instalacao").count()))
print("instalação depois de 2026-06-30:", n_br(datas.filter("instalacao > DATE'2026-06-30'").count()))
print("retirada depois de 2026-06-30:", n_br(datas.filter("retirada > DATE'2026-06-30'").count()))
display(datas.groupBy(F.year("instalacao").alias("ano_instalacao")).count().orderBy("ano_instalacao"))

## 9. Medidores por UC

**O que explora:** quantos medidores ativos cada UC tem. O esperado é 1. Há 45.445 medidores ativos para 46.461 UCs (log, cruzamentos X). Verifica as UCs sem medidor ativo (e o `status` delas no cadastro) e as UCs com mais de um ativo.

**Achado:**
- **45.200 UCs distintas** no cadastro (e não 46.461: o cadastro tem 1.260 linhas repetidas, a investigar na exploração de `unidades_consumidoras`).
- **45.000 UCs com exatamente 1 medidor ativo** e nenhuma com mais de um.
- **200 UCs sem medidor ativo, todas `ATIVA` no cadastro e sem nenhuma leitura** em `silver.leituras`. Não é "consumo sem medidor", e sim UC ativa sem medição nem faturamento. Pode ser erro de cadastro ou UC ligada sem medição (perda). Levado ao log como cruzamento, para decisão do grupo.

In [0]:
ativos = (med_d.filter(F.upper(F.trim("status")) == "ATIVO")
          .groupBy("id_uc").agg(F.count("*").alias("n_ativos")))

ucs = (ucs_bronze.filter(F.col("id_uc") != "id_uc").dropDuplicates(["id_uc"])
       .select("id_uc", F.upper(F.trim("status")).alias("status_uc")))

por_uc = ucs.join(ativos, "id_uc", "left").withColumn("n_ativos", F.coalesce("n_ativos", F.lit(0)))

display(por_uc.groupBy("n_ativos").count().orderBy("n_ativos"))
print("UCs sem medidor ativo, por status da UC:")
display(por_uc.filter("n_ativos = 0").groupBy("status_uc").count())

In [0]:
# UCs sem medidor ativo: têm leitura recente? (consumo sem medidor ativo seria suspeito)
leit = spark.table(f"{silver_schema}.leituras")
ult_leitura = leit.groupBy("id_uc").agg(F.max("mes_referencia").alias("ultima_leitura"))

sem_ativo = por_uc.filter("n_ativos = 0").join(ult_leitura, "id_uc", "left")
display(sem_ativo.groupBy("status_uc", "ultima_leitura").count().orderBy(F.desc("count")).limit(20))

# UCs com mais de um medidor ativo: exemplos
display(med_d.join(por_uc.filter("n_ativos > 1").select("id_uc"), "id_uc").orderBy("id_uc").limit(20))

## 10. Cruzamento com leituras

**O que explora:** a regra de órfãs da Silver de leituras (log L2) compara com os ids distintos de `medidores` da Bronze. A Silver de medidores **não pode remover** nenhum medidor usado em `silver.leituras`, senão os números de leituras mudam. Verifica também se há leituras de um medidor depois da sua retirada.

**Achado:**
- Os 51.299 medidores aparecem em `silver.leituras`, e nenhum medidor usado nas leituras some depois de deduplicar `medidores`. **A Silver de medidores não altera a regra de órfãs de leituras (L2).**
- Nenhum medidor nunca lido, e nenhuma leitura depois do mês de retirada do medidor.

In [0]:
ids_leit = leit.select("id_medidor").distinct()
print("medidores usados em silver.leituras:", n_br(ids_leit.count()))
print("  ausentes na Bronze de medidores (deduplicada):",
      n_br(ids_leit.join(med_d.select("id_medidor"), "id_medidor", "left_anti").count()))
print("medidores nunca lidos:", n_br(med_d.join(ids_leit, "id_medidor", "left_anti").count()))

# Leitura depois do mês de retirada (só retiradas reais em ISO)
ret = (datas.filter("retirada IS NOT NULL")
       .select("id_medidor", F.date_format("retirada", "yyyy-MM").alias("mes_retirada")))
apos = leit.join(ret, "id_medidor").filter("mes_referencia > mes_retirada")
print("leituras depois do mês de retirada do medidor:", n_br(apos.count()))

## 11. Resumo e regras propostas para a Silver

| Problema | Linhas | Regra | Log |
|---|---|---|---|
| `data_ultima_manutencao` 100% vazia | 51.299 | manter (tipo `date`), sem uso em métricas | M1 |
| `data_retirada = 9999-12-31` em ATIVO | 13.500 | vira nulo | M2 |
| Cópias exatas | 513 | `dropDuplicates(["id_medidor"])` | M3 |
| `data_instalacao` em 4 formatos | 3.694 fora do ISO | cada padrão com o seu formato (hífen = mês primeiro) → `date` | M4 |
| `tipo` com 10 grafias | 5.297 | normalizar para `ELETRONICO`, `ELETROMECANICO`, `SMART` | M5 |
| `modelo` com espaços | 3.078 | `trim` | M6 |

### Ordem das operações na Silver
1. Remover as cópias antes de qualquer contagem ou join.
2. Converter a sentinela de `data_retirada` antes do cast de data.
3. Converter `data_instalacao` lendo o padrão do **texto original**.
4. Não remover nenhum medidor: todos são usados em `silver.leituras` (seção 10).

